In [1]:
import sys
import os
sys.path.append(os.path.abspath("../src"))

In [2]:
from pathlib import Path
from abstractionssymh.debug_utils import debug_info, debug_error, debug_success

def get_dataset_directory():
    current_path = Path.cwd()
    base_project_dir = current_path.parent
    dataset_directory = base_project_dir / "src" / "abstractionssymh" / "dataset"

    debug_info("Current notebook location:", current_path)
    debug_info("Base project directory:", base_project_dir)
    debug_info("Target dataset directory:", dataset_directory)

    return dataset_directory

In [3]:
dataset_directory = get_dataset_directory()

[INFO] Current notebook location: /Users/amogh/Projects/abstraction-discovery/notebookssymh
[INFO] Base project directory: /Users/amogh/Projects/abstraction-discovery
[INFO] Target dataset directory: /Users/amogh/Projects/abstraction-discovery/src/abstractionssymh/dataset


In [4]:
import random
from abstractionssymh.data_loader import parse_json_to_dsl

def load_chair_dsl(chair_directory, use_random=False):
    json_files = sorted(chair_directory.glob("*.json"))
    if not json_files:
        debug_error("No JSON files found in:", chair_directory)
        return None

    file_path = random.choice(json_files) if use_random else json_files[0]
    debug_success("Loading chair file:", file_path)

    json_content = file_path.read_text(encoding="utf-8")
    dsl_object = parse_json_to_dsl(json_content)

    debug_success("Successfully parsed DSL object")
    debug_info(dsl_object)
    return dsl_object

In [5]:
chair_directory = dataset_directory / "Chair"
dsl_object = load_chair_dsl(chair_directory, use_random=False)

[SUCCESS] Loading chair file: /Users/amogh/Projects/abstraction-discovery/src/abstractionssymh/dataset/Chair/Chair_1.json
[SUCCESS] Successfully parsed DSL object
[INFO] Union(
    Union(
        Translate(center=[0.003, 0.149, 0.072])
            Rotate(quat=[0.0000, 0.0000, 0.0000, 1.0000])
                Scale(lengths=[0.891, 0.224, 0.806])
                    Box(label=1),
        Translate(center=[0.004, -0.422, 0.170])
            Rotate(quat=[-0.3389, -0.0000, 0.0065, 0.9408])
                Scale(lengths=[0.898, 0.524, 1.154])
                    Box(label=2)
    ),
    Translate(center=[0.030, 0.488, -0.315])
        Rotate(quat=[-0.0821, -0.0071, -0.0494, 0.9954])
            Scale(lengths=[0.892, 0.781, 0.305])
                Box(label=0)
)


In [6]:
from abstractionssymh.plot_utils import plot_dsl_with_k3d

def plot_chair(dsl_obj):
    if dsl_obj is None:
        debug_error("No DSL object to plot.")
        return
    try:
        debug_info("Rendering DSL object with k3d...")
        plot_dsl_with_k3d(dsl_obj)
        debug_success("Plotting complete.")
    except Exception as e:
        debug_error("Failed to plot DSL object:", e)

In [7]:
plot_chair(dsl_object)

[INFO] Rendering DSL object with k3d...
[INFO] Expanding DSL tree for visualization...
[INFO] Found 3 total boxes after expansion.


Output()

[SUCCESS] 3D plot displayed successfully.
[SUCCESS] Plotting complete.


In [8]:
from abstractionssymh.dsl_utils import collect_singleton_and_pair_data

In [9]:
# Load DSL shapes

from abstractionssymh.abstraction_utils import find_abstractions


chair_directory = dataset_directory / "Chair"
json_files = sorted(list(chair_directory.glob("*.json")))

try:
    debug_info("Starting to load DSL shapes from JSON files...")
    all_dsl_shapes = [parse_json_to_dsl(Path(f).read_text()) for f in json_files[:100]]
    debug_success(f"Loaded {len(all_dsl_shapes)} shapes from dataset.")
except Exception as e:
    debug_error("Failed to load DSL shapes:", e)
    all_dsl_shapes = []

# Collect parameters
if all_dsl_shapes:
    debug_info("Collecting singleton and pair parameters from loaded shapes...")
    singleton_params, pair_params = collect_singleton_and_pair_data(all_dsl_shapes)

    num_singleton = sum(len(v) for v in singleton_params.values())
    num_pair = sum(len(v) for v in pair_params.values())
    debug_success(f"Collected {num_singleton} singleton and {num_pair} pair parameter sets.")

    # Report top structures by count
    # top_singletons = sorted(singleton_params.items(), key=lambda x: len(x[1]), reverse=True)[:5]
    # top_pairs = sorted(pair_params.items(), key=lambda x: len(x[1]), reverse=True)[:5]
    # debug_info("Top singleton structures by example count:", top_singletons)
    # debug_info("Top pair structures by example count:", top_pairs)

    # Train singleton models
    debug_info("Starting singleton model training...")
    singleton_models = find_abstractions(singleton_params, structure_type="SINGLETONS", min_examples=50, epochs=25)

    # Train pair models
    debug_info("Starting pair model training...")
    pair_models = find_abstractions(pair_params, structure_type="PAIRS", min_examples=50, epochs=25)

    debug_success(f"Training complete. {len(singleton_models)} singleton models and {len(pair_models)} pair models trained.")

else:
    debug_error("No DSL shapes available. Abstraction pipeline skipped.")

[INFO] Starting to load DSL shapes from JSON files...
[SUCCESS] Loaded 100 shapes from dataset.
[INFO] Collecting singleton and pair parameters from loaded shapes...
[SUCCESS] Collected keys: dict_keys(['Scale', 'Rotate', 'Translate', 'SymRef', 'SymRot', 'SymTrans']) singletons, dict_keys(['Scale(Box)', 'Rotate(Scale)', 'Translate(Rotate)', 'Union(Translate)', 'SymRef(Union)', 'Union(SymRef)', 'SymRef(Translate)', 'SymRot(Union)', 'Union(SymRot)', 'SymRot(Translate)', 'SymTrans(Translate)', 'Union(SymTrans)']) pairs
[SUCCESS] Collected parameters: 6 singletons, 12 pairs
[SUCCESS] Collected 2196 singleton and 2984 pair parameter sets.
[INFO] Starting singleton model training...
[INFO] Finding abstractions for 6 structures of type SINGLETONS
[SUCCESS] Training model for Rotate with 662 samples, 4 params
[INFO] Iteration 1/3 for Rotate
[INFO] Preparing DataLoader: batch_size=64, mask sum=662
[SUCCESS] DataLoader prepared successfully.
[INFO] Initializing Autoencoder: input_dim=4, hidden_d

In [10]:
from abstractionssymh.abstraction_utils import Abstraction, integrate_abstractions

# Test on a random DSL shape
if all_dsl_shapes and singleton_models and pair_models:
    random_chair = random.choice(all_dsl_shapes)
    debug_info("--- ORIGINAL CHAIR ---")
    debug_info(f"Type: {type(random_chair).__name__}")
    debug_info(f"Serialized children count: {len(random_chair.serialize()[1][1])}")
    debug_info(f"Preview: {random_chair}")

    # Integrate abstractions
    debug_info("Integrating abstractions...")
    abstracted_chair = integrate_abstractions(
        random_chair, singleton_models, pair_models, error_threshold=0.01
    )

    debug_info("\n--- ABSTRACTED CHAIR ---")
    debug_info(f"Type: {type(abstracted_chair).__name__}")
    if isinstance(abstracted_chair, Abstraction):
        debug_info(f"Abstraction pattern: {abstracted_chair.pattern_name}, compressed dim: {len(abstracted_chair.compressed_params)}")
    debug_info(f"Preview: {abstracted_chair}")

    # Visualization
    try:
        debug_info("Plotting original and abstracted DSL shapes...")
        plot_dsl_with_k3d(random_chair)
        plot_dsl_with_k3d(abstracted_chair)
        debug_success("Visualization complete.")
    except Exception as e:
        debug_error("Plotting failed:", e)
else:
    debug_error("Cannot run test: DSL shapes or trained models are missing.")

[INFO] --- ORIGINAL CHAIR ---
[INFO] Type: Union
[INFO] Serialized children count: 2
[INFO] Preview: Union(
    Union(
        Translate(center=[0.000, -0.191, 0.095])
            Rotate(quat=[0.0000, 0.0000, 0.0000, 1.0000])
                Scale(lengths=[0.921, 0.356, 0.921])
                    Box(label=1),
        SymRef(
            plane=[1.000, -0.000, 0.011],
            point_on_plane=[-0.004, -0.564, -0.300]
        )(
            Union(
                Translate(center=[-0.400, -0.564, -0.305])
                    Rotate(quat=[0.0000, 0.0000, 0.0000, 1.0000])
                        Scale(lengths=[0.121, 0.452, 0.120])
                            Box(label=2),
                Translate(center=[-0.400, -0.564, 0.487])
                    Rotate(quat=[0.0000, 0.0000, 0.0000, 1.0000])
                        Scale(lengths=[0.121, 0.452, 0.135])
                            Box(label=2)
            )
        )
    ),
    Translate(center=[0.000, 0.349, -0.119])
        Rotate(qu

Output()

[SUCCESS] 3D plot displayed successfully.
[INFO] Expanding DSL tree for visualization...
[INFO] Expanding abstraction: Abs(Rotate, dim=3)(
    Scale(lengths=[0.921, 0.356, 0.921])
        Box(label=1)
)
[INFO] Compressed params tensor shape: (1, 3)
[SUCCESS] Reconstructed params: [-0.001698957523331046, -0.0005006976425647736, -0.0001614391803741455, 0.9998127222061157]
[INFO] [INSTANTIATE] Pattern: 'Rotate' | params=[-0.001698957523331046, -0.0005006976425647736, -0.0001614391803741455, 0.9998127222061157] | #children=1
[SUCCESS] Successfully rebuilt node: Rotate(quat=[-0.0017, -0.0005, -0.0002, 0.9998])
    Scale(lengths=[0.921, 0.356, 0.921])
        Box(label=1)
[INFO] Expanding abstraction: Abs(Rotate, dim=3)(
    Scale(lengths=[0.121, 0.452, 0.120])
        Box(label=2)
)
[INFO] Compressed params tensor shape: (1, 3)
[SUCCESS] Reconstructed params: [-0.001698957523331046, -0.0005006976425647736, -0.0001614391803741455, 0.9998127222061157]
[INFO] [INSTANTIATE] Pattern: 'Rotate' | 

Output()

[SUCCESS] 3D plot displayed successfully.
[SUCCESS] Visualization complete.


In [11]:
print(abstracted_chair)

Union(
    Union(
        Translate(center=[0.000, -0.191, 0.095])
            Abs(Rotate, dim=3)(
                Scale(lengths=[0.921, 0.356, 0.921])
                    Box(label=1)
            ),
        SymRef(
            plane=[1.000, -0.000, 0.011],
            point_on_plane=[-0.004, -0.564, -0.300]
        )(
            Union(
                Translate(center=[-0.400, -0.564, -0.305])
                    Abs(Rotate, dim=3)(
                        Scale(lengths=[0.121, 0.452, 0.120])
                            Box(label=2)
                    ),
                Translate(center=[-0.400, -0.564, 0.487])
                    Abs(Rotate, dim=3)(
                        Scale(lengths=[0.121, 0.452, 0.135])
                            Box(label=2)
                    )
            )
        )
    ),
    Translate(center=[0.000, 0.349, -0.119])
        Abs(Rotate, dim=3)(
            Scale(lengths=[0.921, 0.827, 0.493])
                Box(label=0)
        )
)


In [12]:
print(random_chair)

Union(
    Union(
        Translate(center=[0.000, -0.191, 0.095])
            Rotate(quat=[0.0000, 0.0000, 0.0000, 1.0000])
                Scale(lengths=[0.921, 0.356, 0.921])
                    Box(label=1),
        SymRef(
            plane=[1.000, -0.000, 0.011],
            point_on_plane=[-0.004, -0.564, -0.300]
        )(
            Union(
                Translate(center=[-0.400, -0.564, -0.305])
                    Rotate(quat=[0.0000, 0.0000, 0.0000, 1.0000])
                        Scale(lengths=[0.121, 0.452, 0.120])
                            Box(label=2),
                Translate(center=[-0.400, -0.564, 0.487])
                    Rotate(quat=[0.0000, 0.0000, 0.0000, 1.0000])
                        Scale(lengths=[0.121, 0.452, 0.135])
                            Box(label=2)
            )
        )
    ),
    Translate(center=[0.000, 0.349, -0.119])
        Rotate(quat=[0.0000, 0.0000, 0.0000, 1.0000])
            Scale(lengths=[0.921, 0.827, 0.493])
               